In [ ]:
import pandas as pd
import os
from helper_functions import strings2lists

# df_path = r"D:\DATA\audit\mpp_audit_slides.csv"
df_path = r"D:\DATA\abmil_audit_v2.csv"

df = pd.read_csv(df_path)

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df[col] = df[col].apply(strings2lists)

print("Tissue distribution:\n", df["_tissue_rep"].value_counts())

zarr_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"

In [ ]:
from abmil_audit import KFoldPipeline

filename_col = "filename"
label_col = "M_idx"
patient_col = "rekvnr"
n_splits = 5
n_epochs = 30
early_stopping_patience = 5
max_tiles = 10000
random_state = 42

In [ ]:
# CV pipeline for mpp 0.12
cv_012 = KFoldPipeline(
    df=df,
    filename_col=filename_col,
    label_col=label_col,
    patient_col = patient_col,
    feature_key="features_h-optimus-0",
    tile_key="tiles_224",
    zarr_dir=zarr_dir,
)

In [ ]:
cv_012.validate_slides()

# Dataset validation complete: 116/120 valid slides
# Validation complete: 116 valid slides (removed 4)

In [ ]:
dir_012 = r"D:\DATA\audit\v2\checkpoints_mpp0.12"

results_012 = cv_012.kfold_cross_validation(
    n_splits=n_splits,
    n_epochs=n_epochs,
    early_stopping_patience=early_stopping_patience,
    max_tiles=max_tiles,
    random_state=42,
    resume_from_checkpoints=True,
    checkpoint_dir=dir_012,
)

In [ ]:
cv_012.print_results()

In [ ]:
# CV pipeline for mpp 0.5
cv_05 = KFoldPipeline(
    df=df,
    filename_col=filename_col,
    label_col=label_col,
    patient_col = patient_col,
    feature_key="features_h-optimus-0_mpp0.5",
    tile_key="tiles_224_mpp0.5",
    zarr_dir=zarr_dir,
)

In [ ]:
cv_05.validate_slides()

# Dataset validation complete: 195/200 valid slides
# Validation complete: 195 valid slides (removed 5)

In [ ]:
dir_05 = r"D:\DATA\audit\v2\checkpoints_mpp0.5"

results_05 = cv_05.kfold_cross_validation(
    n_splits=n_splits,
    n_epochs=n_epochs,
    early_stopping_patience=early_stopping_patience,
    max_tiles=max_tiles,
    random_state=42,
    resume_from_checkpoints=True,
    checkpoint_dir=dir_05,
)

In [ ]:
cv_05.print_results()

In [ ]:
from abmil_audit import run_deletion_curve_evaluation, plot_deletion_curves, deletion_auc_summary

# mpp 0.5
results_dc_05 = run_deletion_curve_evaluation(
    df=df, filename_col=filename_col, label_col=label_col, patient_col = patient_col,
    feature_key="features_h-optimus-0_mpp0.5",
    zarr_dir=zarr_dir,
    checkpoint_dir=dir_05,
    n_splits=n_splits, random_state=random_state,
    rank_by="attention",
)
plot_deletion_curves(results_dc_05, title="Deletion curve - MPP 0.5")
audc_05 = deletion_auc_summary(results_dc_05)


In [ ]:
# mpp 0.5
results_dc_05 = run_deletion_curve_evaluation(
    df=df, filename_col=filename_col, label_col=label_col, patient_col = patient_col,
    feature_key="features_h-optimus-0_mpp0.5",
    zarr_dir=zarr_dir,
    checkpoint_dir=dir_05,
    n_splits=n_splits, random_state=random_state, 
    rank_by="contribution_score",
)
plot_deletion_curves(results_dc_05, title="Deletion curve - MPP 0.5")
audc_05 = deletion_auc_summary(results_dc_05)


In [ ]:
# mpp 0.12
results_dc_012 = run_deletion_curve_evaluation(
    df=df, filename_col=filename_col, label_col=label_col, patient_col = patient_col,
    feature_key="features_h-optimus-0",
    zarr_dir=zarr_dir,
    checkpoint_dir=r"D:\DATA\audit\checkpoints_mpp0.12",
    n_splits=n_splits, random_state=random_state,
)
plot_deletion_curves(results_dc_012, title="Deletion curve - MPP 0.12")
audc_012 = deletion_auc_summary(results_dc_012)

In [ ]:
from tissue_artifact_segmentation import _load_cache, _get_processed_entries
from datetime import timedelta

cache_path = r"D:\DATA\cache_tissue_artifact_features.pkl"
cache = _load_cache(cache_path)
entries = _get_processed_entries(cache)

paths = set(df["filename"].tolist())

groups = {
    "mpp0.12": ("features_h-optimus-0", "tiles_224"),
    "mpp0.5": ("features_h-optimus-0_mpp0.5", "tiles_224_mpp0.5"),
}

summary = {}

for label, (feature_key, tile_key) in groups.items():
    bagsizes = []
    elapsed_times = []
    count = 0

    for (_, category, _), entry in entries.items():
        if category != "features":
            continue
        if entry.get("feature_key") != feature_key:
            continue
        if entry.get("tile_key") != tile_key:
            continue
        if entry.get("wsi_path") not in paths:
            continue
        if entry.get("status") == "error":
            continue

        n_tiles = entry.get("n_tiles")
        elapsed_time = entry.get("elapsed_time")
        if n_tiles is None or elapsed_time is None:
            continue

        if isinstance(elapsed_time, timedelta):
            elapsed_time = elapsed_time.total_seconds()

        bagsizes.append(int(n_tiles))
        elapsed_times.append(float(elapsed_time))
        count += 1

    summary[label] = {
        "count": count,
        "median_bagsize": float(np.median(bagsizes)) if bagsizes else np.nan,
        "mean_elapsed_time_seconds": float(np.mean(elapsed_times)) if elapsed_times else np.nan,
    }

for label, stats in summary.items():
    print(
        f"{label}: count={stats['count']}, "
        f"median_bagsize={stats['median_bagsize']:.1f}, "
        f"mean_elapsed_time_seconds={stats['mean_elapsed_time_seconds']:.2f}"
    )